# Module 2 — Malthus, Ricardo, and the Limits to Growth
## Population, Diminishing Returns, and the Malthusian Trap

**From Smith to Simulation: Computing the Ideas that Built Economics**  
The University of Edinburgh · School of Economics

---

## Part I — The History

### The Darkest Idea in Economics

In 1798 — just twenty-two years after Smith's optimistic vision of markets creating wealth — a young English clergyman published one of the most influential and controversial books in the history of social science. His name was **Thomas Robert Malthus** (1766–1834), and his *Essay on the Principle of Population* argued that humanity was trapped.

The logic was brutally simple:

1. **Population**, when unchecked, grows **geometrically** (exponentially) — each generation is larger than the last.
2. **Food production** can only grow **arithmetically** (linearly) — you can clear new fields, but there's only so much land.
3. Therefore, population will always tend to outrun the food supply. The result is misery: famine, disease, and war keep population in check.

> *"The power of population is indefinitely greater than the power in the earth to produce subsistence for man."*  
> — Thomas Malthus, *Essay on the Principle of Population* (1798), Chapter 1

This is why Thomas Carlyle called economics "the dismal science" — though the phrase was actually aimed at Malthus's followers, not Smith.

### Was Malthus Right?

For most of human history, **yes**. For thousands of years before the Industrial Revolution, living standards barely changed. When agricultural productivity improved — a new crop, a better plough — the result was not richer people but *more* people. The population expanded until it pressed against the food supply again, and wages returned to subsistence. Historians call this the **Malthusian trap**.

Gregory Clark, in *A Farewell to Alms* (2007), estimated that the average English person in 1800 was no better off than the average person in 100,000 BC. The Malthusian mechanism explains why: any surplus was eaten up (literally) by population growth.

But then something extraordinary happened. Starting around 1800 in Britain, productivity began to grow faster than population. The Industrial Revolution broke the Malthusian trap — at least for some countries. Understanding *why* is one of the great questions of economics, and it's what makes the Malthusian model so interesting: it describes the world accurately for 99% of human history, and then suddenly stops working.

### David Ricardo and Diminishing Returns

Malthus's friend and intellectual sparring partner was **David Ricardo** (1772–1823), a stockbroker turned economist who formalised many of the ideas that Malthus and Smith had expressed in words.

Ricardo's key contribution to this story is the concept of **diminishing returns**. As more labour is applied to a fixed amount of land, each additional worker produces less additional output. The first farmers clear the best land; the next clear the second-best; eventually you're farming rocky hillsides. This is why food production can't keep up with population growth — not because farming doesn't improve, but because the *marginal* improvement gets smaller.

Ricardo also showed how diminishing returns determined the **distribution of income** between workers (who receive wages), landlords (who receive rent), and capitalists (who receive profits). As population grows and worse land is brought into cultivation, landlords capture an increasing share of national income through rising rents. Workers are stuck at subsistence. It's a grim picture — and Marx would later build on it.

### Edinburgh Connection

Malthus's *Essay* was partly inspired by the Marquis de Condorcet and William Godwin, Enlightenment optimists who believed humanity was perfectible. Malthus — educated in the Scottish tradition of empirical argument — responded with data and logic rather than hope. The Scottish Enlightenment's commitment to following evidence wherever it leads, even to uncomfortable conclusions, is visible throughout Malthus's work. Dugald Stewart, who held Adam Smith's old chair of moral philosophy at Edinburgh, was one of the key figures who disseminated both Smith's and Malthus's ideas to the next generation.

---

## Part II — The Computation

### Setting Up

We'll build and simulate the Malthusian model step by step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### The Production Function: Diminishing Returns

Following Ricardo, we model food production with a **Cobb-Douglas production function** where land is fixed:

$$Y = A \cdot L^\alpha$$

Here $Y$ is total food output, $L$ is the population (which equals the labour force in this pre-industrial world), $A$ is a productivity parameter (think: quality of seeds, farming techniques), and $\alpha \in (0, 1)$ captures diminishing returns.

The crucial feature is that $\alpha < 1$: doubling the population does *not* double food production. This is Ricardo's diminishing returns in action.

In [ ]:
# Parameters
A = 10       # productivity
alpha = 0.6  # diminishing returns (less than 1)

def production(L, A=A, alpha=alpha):
    """Total food output given population L."""
    return A * L**alpha

def output_per_capita(L, A=A, alpha=alpha):
    """Food per person = Y/L = A * L^(alpha-1)."""
    return A * L**(alpha - 1)

# Visualise
L_range = np.linspace(1, 500, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(L_range, production(L_range), color='seagreen', linewidth=2)
axes[0].set_xlabel('Population (L)', fontsize=12)
axes[0].set_ylabel('Total output (Y)', fontsize=12)
axes[0].set_title('Total Production: Diminishing Returns', fontsize=13)
axes[0].grid(True, alpha=0.3)

axes[1].plot(L_range, output_per_capita(L_range), color='coral', linewidth=2)
axes[1].set_xlabel('Population (L)', fontsize=12)
axes[1].set_ylabel('Output per person (Y/L)', fontsize=12)
axes[1].set_title('Output per Capita Falls with Population', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice: total output rises with population, but output per person falls.")
print("This is the essence of the Malthusian trap.")

### Population Dynamics: The Malthusian Mechanism

Malthus's key insight is that population growth *responds* to living standards. When food per person is above subsistence, people have more children and fewer die — population grows. When food per person falls below subsistence, famine and disease cause population to shrink.

We model this as:

$$L_{t+1} = L_t \cdot \left(1 + g\left(\frac{Y_t}{L_t} - \bar{c}\right)\right)$$

where:
- $Y_t / L_t$ is food per person at time $t$
- $\bar{c}$ is the subsistence level of consumption
- $g$ controls how strongly population responds to the gap between actual and subsistence consumption

When $Y/L > \bar{c}$: population grows.  
When $Y/L < \bar{c}$: population shrinks.  
When $Y/L = \bar{c}$: population is stable — the **Malthusian steady state**.

In [ ]:
# Model parameters
c_bar = 1.5    # subsistence consumption
g = 0.02       # population growth sensitivity
T = 300        # number of periods to simulate
L0 = 50        # initial population

def simulate_malthus(L0, A, alpha, c_bar, g, T):
    """
    Simulate the Malthusian model.
    Returns arrays of population and output per capita.
    """
    L = np.empty(T)
    y = np.empty(T)  # output per capita
    L[0] = L0
    y[0] = output_per_capita(L0, A, alpha)
    
    for t in range(1, T):
        # Population growth depends on gap from subsistence
        growth_rate = g * (y[t-1] - c_bar)
        L[t] = L[t-1] * (1 + growth_rate)
        L[t] = max(L[t], 1)  # population can't go below 1
        y[t] = output_per_capita(L[t], A, alpha)
    
    return L, y

# Run the simulation
L_sim, y_sim = simulate_malthus(L0, A, alpha, c_bar, g, T)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(L_sim, color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Time', fontsize=12)
axes[0].set_ylabel('Population', fontsize=12)
axes[0].set_title('Population Over Time', fontsize=13)
axes[0].grid(True, alpha=0.3)

axes[1].plot(y_sim, color='coral', linewidth=1.5)
axes[1].axhline(c_bar, color='black', linestyle='--', linewidth=1, label=f'Subsistence = {c_bar}')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Output per capita', fontsize=12)
axes[1].set_title('Living Standards: Trapped at Subsistence', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Steady-state population: {L_sim[-1]:.1f}")
print(f"Steady-state output per capita: {y_sim[-1]:.4f}")
print(f"Subsistence level: {c_bar}")

Look at what happens: the population grows until output per capita falls to the subsistence level, then stabilises. Living standards are trapped — exactly as Malthus predicted.

### The Cruel Irony: Does Better Technology Help?

Here's the most counterintuitive implication of the Malthusian model. Suppose someone invents a better plough that increases productivity ($A$). You might expect this to make everyone richer. But watch what actually happens:

In [ ]:
# Compare three productivity levels
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for A_val, color, label in [(8, 'steelblue', 'A = 8 (poor technology)'),
                             (10, 'seagreen', 'A = 10 (baseline)'),
                             (15, 'coral', 'A = 15 (better plough!)')]:
    L_s, y_s = simulate_malthus(L0, A_val, alpha, c_bar, g, T)
    axes[0].plot(L_s, color=color, linewidth=1.5, label=label)
    axes[1].plot(y_s, color=color, linewidth=1.5, label=label)

axes[0].set_xlabel('Time', fontsize=12)
axes[0].set_ylabel('Population', fontsize=12)
axes[0].set_title('Population', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].axhline(c_bar, color='black', linestyle='--', linewidth=1, label='Subsistence')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Output per capita', fontsize=12)
axes[1].set_title('Living Standards', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The cruel irony: better technology leads to MORE PEOPLE, not RICHER people.")
print("In the long run, all three economies converge to the same subsistence living standard.")

This is the Malthusian trap in its starkest form. A technological improvement temporarily raises living standards, but population grows in response, and eventually living standards fall right back to subsistence. The only lasting effect is a larger population. Better technology produces more people, not richer people.

This is why, for most of history, China and India had the world's largest populations but not the highest living standards — they had productive agriculture, which translated into *more mouths* rather than *fuller plates*.

### Finding the Steady State Analytically

At the Malthusian steady state, output per capita equals subsistence:

$$\frac{Y}{L} = A \cdot L^{\alpha - 1} = \bar{c}$$

Solving for $L^*$:

$$L^* = \left(\frac{A}{\bar{c}}\right)^{\frac{1}{1-\alpha}}$$

Let's verify our simulation matches this formula:

In [ ]:
def malthus_steady_state(A, alpha, c_bar):
    """Analytical steady-state population."""
    return (A / c_bar) ** (1 / (1 - alpha))

L_star = malthus_steady_state(A, alpha, c_bar)
print(f"Analytical steady-state population: {L_star:.2f}")
print(f"Simulation steady-state population: {L_sim[-1]:.2f}")
print(f"Match: {np.isclose(L_star, L_sim[-1], rtol=0.01)}")

# Effect of better technology on steady-state population
print("\nEffect of productivity on steady-state population:")
for A_val in [8, 10, 12, 15, 20]:
    L_ss = malthus_steady_state(A_val, alpha, c_bar)
    print(f"  A = {A_val:>3} → L* = {L_ss:>8.1f}")

print("\nBetter technology → bigger population, but SAME living standard (subsistence).")

### Breaking the Trap: The Industrial Revolution

The Malthusian model describes the pre-industrial world perfectly. But after 1800, something changed — productivity began growing *continuously* and *faster* than population could keep up. Let's model this by making $A$ grow over time:

$$A_t = A_0 \cdot e^{\gamma t}$$

where $\gamma$ is the rate of technological progress. If $\gamma$ is large enough, the economy escapes the trap.

In [ ]:
def simulate_malthus_with_growth(L0, A0, alpha, c_bar, g, gamma, T):
    """
    Malthusian model with technological progress.
    gamma = rate of productivity growth.
    """
    L = np.empty(T)
    y = np.empty(T)
    A_t = np.empty(T)
    
    L[0] = L0
    A_t[0] = A0
    y[0] = A0 * L0**(alpha - 1)
    
    for t in range(1, T):
        A_t[t] = A0 * np.exp(gamma * t)
        growth_rate = g * (y[t-1] - c_bar)
        L[t] = L[t-1] * (1 + growth_rate)
        L[t] = max(L[t], 1)
        y[t] = A_t[t] * L[t]**(alpha - 1)
    
    return L, y, A_t

# Compare: no growth, slow growth, Industrial Revolution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

scenarios = [
    (0.000, 'steelblue', 'No tech. growth (pre-1800)'),
    (0.003, 'seagreen',  'Slow growth (γ = 0.3%)'),
    (0.010, 'coral',     'Industrial Revolution (γ = 1%)')
]

for gamma, color, label in scenarios:
    L_s, y_s, _ = simulate_malthus_with_growth(L0, A, alpha, c_bar, g, gamma, 400)
    axes[0].plot(L_s, color=color, linewidth=1.5, label=label)
    axes[1].plot(y_s, color=color, linewidth=1.5, label=label)

axes[0].set_xlabel('Time', fontsize=12)
axes[0].set_ylabel('Population', fontsize=12)
axes[0].set_title('Population', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].axhline(c_bar, color='black', linestyle='--', linewidth=0.8, label='Subsistence')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Output per capita', fontsize=12)
axes[1].set_title('Living Standards', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("With fast enough technological progress, living standards break free from subsistence.")
print("This is what happened in Britain around 1800 — the escape from the Malthusian trap.")

This is one of the most important transitions in human history, and you've just simulated it. For 10,000 years, the world was stuck in the flat part of the curve. Then, in a few decades, everything changed.

The question of *why* it happened — why in Britain, why around 1800, and not somewhere else or some other time — remains one of the most debated questions in economic history.

---

## Part III — Exercises

### Exercise 1 — The Black Death

In 1348, the Black Death killed roughly one-third of Europe's population. The Malthusian model makes a clear prediction about what should happen next.

**(a)** Simulate the Malthusian model for 500 periods with $L_0 = 200$, $A = 10$, $\alpha = 0.6$, $\bar{c} = 1.5$, $g = 0.02$. Let the economy reach its steady state.

**(b)** At period 250, kill one-third of the population (multiply $L$ by $2/3$). Continue the simulation for the remaining periods. Plot population and output per capita.

**(c)** What happens to living standards immediately after the plague? What happens in the long run? Does this match the historical evidence? (Hint: real wages in England roughly doubled after the Black Death.)

**(d)** In 2–3 sentences, explain why a catastrophe that kills millions can *improve* living standards for survivors, using the model's logic.

In [ ]:
# Your answer here


### Exercise 2 — Ricardo's Income Distribution

Ricardo argued that as population grows, landlords capture an increasing share of national income. Let's model this.

With production $Y = A \cdot L^\alpha$, the **marginal product of labour** is:

$$MPL = \alpha \cdot A \cdot L^{\alpha - 1}$$

In a competitive labour market, the wage equals the marginal product: $w = MPL$.

- **Total wages** = $w \cdot L = \alpha \cdot Y$
- **Rents** (what goes to landlords) = $Y - w \cdot L = (1 - \alpha) \cdot Y$
- **Labour share** = $\alpha$ (constant in this model, but the *level* of wages falls as $L$ rises)

**(a)** Compute and plot the wage $w$ and total rents $(1 - \alpha) Y$ as functions of population $L$ for $L \in [10, 500]$. Use $A = 10$, $\alpha = 0.6$.

**(b)** As population doubles from 100 to 200, what happens to individual wages? What happens to total rents? Who benefits from population growth — workers or landlords?

**(c)** Compute the **rent-to-wage ratio** $(1-\alpha)Y / (\alpha \cdot A \cdot L^{\alpha-1})$ as a function of $L$. Plot it. What happens as population grows?

In [ ]:
# Your answer here


### Exercise 3 — Malthus in the Modern World

Some argue that Malthusian dynamics are still relevant today — not for food, but for other scarce resources (energy, water, arable land, the atmosphere's capacity to absorb CO2).

**(a)** Modify the Malthusian simulation to include a **carrying capacity** for the environment. Let output be:

$$Y = A \cdot L^\alpha \cdot \left(1 - \frac{L}{K}\right)$$

where $K$ is the maximum population the environment can support. (When $L$ approaches $K$, output drops to zero.) Simulate with $K = 500$, $A = 10$, $\alpha = 0.6$, $L_0 = 50$.

**(b)** What happens to the steady-state population and living standards compared to the basic Malthusian model?

**(c)** Now add technological progress ($\gamma = 0.005$). Does the economy still escape the trap? What is different from the model without the carrying capacity?

In [ ]:
# Your answer here


---

## Part IV — Quiz

### Conceptual Questions

**Q1.** Malthus argued that population grows geometrically while food production grows arithmetically. In modern terms, this means:

(a) Population grows linearly, food grows exponentially  
(b) Population grows exponentially, food grows linearly  
(c) Both grow at the same rate  
(d) Both grow exponentially but at different rates

**Q2.** In the Malthusian model, a one-time improvement in agricultural productivity (higher $A$) leads to:

(a) Permanently higher living standards  
(b) Temporarily higher living standards, then a return to subsistence with a larger population  
(c) Lower living standards because more food attracts invaders  
(d) No change in either population or living standards

**Q3.** Ricardo's concept of "diminishing returns" means:

(a) Profits fall to zero in the long run  
(b) Each additional unit of labour applied to fixed land produces less additional output than the one before  
(c) Wages always decrease over time  
(d) Trade between countries always produces diminishing benefits

**Q4.** Why did Thomas Carlyle call economics "the dismal science"?

(a) Because economists couldn't agree on anything  
(b) Because of the pessimistic predictions of Malthus and his followers  
(c) Because Smith's model predicted market crashes  
(d) Because Ricardo proved that trade was harmful

**Q5.** The Malthusian trap was broken by:

(a) The discovery of the Americas  
(b) The Black Death  
(c) The Industrial Revolution, where productivity growth outpaced population growth  
(d) Government population controls

### Computational Questions

**Q6.** In the production function $Y = A \cdot L^{0.6}$, if the population doubles from 100 to 200, total output:

(a) Exactly doubles  
(b) More than doubles  
(c) Increases by a factor of $2^{0.6} \approx 1.52$ (less than doubles)  
(d) Stays the same

**Q7.** In the Malthusian steady state, output per capita equals:

(a) Zero  
(b) The subsistence level $\bar{c}$  
(c) The productivity parameter $A$  
(d) The growth rate $g$

**Q8.** The steady-state population in the Malthusian model is $L^* = (A/\bar{c})^{1/(1-\alpha)}$. If $A$ doubles, the steady-state population:

(a) Doubles  
(b) More than doubles  
(c) Less than doubles  
(d) Depends on whether $\alpha > 0.5$

**Q9.** If the Black Death kills half the population, the Malthusian model predicts that living standards will:

(a) Fall immediately and stay low  
(b) Rise immediately, then gradually return to subsistence as population recovers  
(c) Stay unchanged because total output also falls by half  
(d) Rise permanently

**Q10.** In the simulation with technological progress, the economy escapes the Malthusian trap when:

(a) $A$ becomes very large  
(b) Productivity grows faster than population can respond  
(c) The subsistence level falls to zero  
(d) $\alpha$ exceeds 1

### Quiz Answers

<details>
<summary>Click to reveal answers</summary>

**Q1.** (b) Geometric growth is exponential; arithmetic growth is linear. Malthus's whole argument hinges on this asymmetry.

**Q2.** (b) Higher $A$ temporarily raises $Y/L$ above $\bar{c}$, causing population growth, which continues until $Y/L$ returns to $\bar{c}$. The steady-state population is higher, but living standards are the same.

**Q3.** (b) Diminishing returns means each additional worker on the same amount of land adds less to total output. This is why $\alpha < 1$ in $Y = A L^\alpha$.

**Q4.** (b) The phrase "dismal science" came from the gloomy predictions of Malthus and his followers that humanity was doomed to subsistence.

**Q5.** (c) The Industrial Revolution brought sustained technological progress that outpaced population growth, breaking the trap for the first time in human history.

**Q6.** (c) With $\alpha = 0.6$, doubling $L$ multiplies output by $2^{0.6} \approx 1.52$. Output rises but by less than double — that's diminishing returns.

**Q7.** (b) The steady state is defined by $Y/L = \bar{c}$. Population adjusts until per capita output exactly equals subsistence.

**Q8.** (b) $L^* = (A/\bar{c})^{1/(1-\alpha)}$. With $\alpha = 0.6$, doubling $A$ multiplies $L^*$ by $2^{1/0.4} = 2^{2.5} \approx 5.66$, which is more than double.

**Q9.** (b) With fewer people sharing the same land and technology, $Y/L$ jumps above $\bar{c}$. But this triggers population growth, which gradually pushes $Y/L$ back to subsistence. This is exactly what happened historically: English real wages roughly doubled after the Black Death, then slowly declined over the next two centuries.

**Q10.** (b) When $A$ grows continuously and fast enough, output per capita keeps rising faster than population can expand to absorb the gains. The demographic transition (falling birth rates as incomes rise) reinforces the escape.

</details>

---

## Further Reading

- Malthus, T.R. *An Essay on the Principle of Population* (1798), Chapter 1. Available free online.
- Clark, G. *A Farewell to Alms: A Brief Economic History of the World* (2007), Chapters 1–2.
- Heilbroner, R. *The Worldly Philosophers*, Chapter 4 ("The Gloomy Presentiments of Parson Malthus and David Ricardo").
- Galor, O. *Unified Growth Theory* (2011) — the modern theoretical framework for understanding the escape from the Malthusian trap.

---

*Next week: Marx and the Machinery Question — capital accumulation, the labour share, and whether technology helps or hurts workers.*